Used dataset: student_data.csv

Fetched from: https://www.kaggle.com/datasets/devansodariya/student-performance-data/data

Classifies student performance based on their average exam scores.

In [17]:
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

In [18]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [19]:
ROOT_DIR = Path.cwd().parent
RAW_DATA_PATH = ROOT_DIR/"data/raw/student_data.csv"
PROCESSED_DATA_DIR = ROOT_DIR/"data/processed"

TRAIN_SIZE = 0.9
TEST_SIZE = 0.1

Load dataset.

In [20]:
df = pd.read_csv(RAW_DATA_PATH, encoding="utf-8")

df.head(3)

,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,reason,guardian,traveltime,studytime,failures,schoolsup,famsup,paid,activities,nursery,higher,internet,romantic,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,course,mother,2,2,0,yes,no,no,no,yes,yes,no,no,4,3,4,1,1,3,6,5,6,6
1,GP,F,17,U,GT3,T,1,1,at_home,other,course,father,1,2,0,no,yes,no,no,no,yes,yes,no,5,3,3,1,1,3,4,5,5,6
2,GP,F,15,U,LE3,T,1,1,at_home,other,other,mother,1,2,3,yes,no,yes,no,yes,yes,yes,no,4,3,2,2,3,3,10,7,8,10


In [21]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 395 entries, 0 to 394
Data columns (total 33 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   school      395 non-null    object
 1   sex         395 non-null    object
 2   age         395 non-null    int64 
 3   address     395 non-null    object
 4   famsize     395 non-null    object
 5   Pstatus     395 non-null    object
 6   Medu        395 non-null    int64 
 7   Fedu        395 non-null    int64 
 8   Mjob        395 non-null    object
 9   Fjob        395 non-null    object
 10  reason      395 non-null    object
 11  guardian    395 non-null    object
 12  traveltime  395 non-null    int64 
 13  studytime   395 non-null    int64 
 14  failures    395 non-null    int64 
 15  schoolsup   395 non-null    object
 16  famsup      395 non-null    object
 17  paid        395 non-null    object
 18  activities  395 non-null    object
 19  nursery     395 non-null    object
 20  higher    

In [22]:
df.duplicated().sum()

np.int64(0)

Quantize average score into low, medium, and high.

In [24]:
df["performance_category"] = pd.qcut(
    x=df[['G1', 'G2', 'G3']].mean(axis=1), 
    q=3, 
    labels=["low", "medium", "high"]
)

Distribution per class.

In [26]:
df['performance_category'].value_counts()

performance_category
medium    136
low       132
high      127
Name: count, dtype: int64

Split feature-target.

In [27]:
X = df.drop(columns=['G1', 'G2', 'G3', "performance_category"])
y = df["performance_category"]

In [28]:
X.describe()    # include number

,age,Medu,Fedu,traveltime,studytime,failures,famrel,freetime,goout,Dalc,Walc,health,absences
count,395.000000,395.000000,395.000000,395.000000,395.000000,395.000000,395.000000,395.000000,395.000000,395.000000,395.000000,395.000000,395.000000
mean,16.696203,2.749367,2.521519,1.448101,2.035443,0.334177,3.944304,3.235443,3.108861,1.481013,2.291139,3.554430,5.708861
std,1.276043,1.094735,1.088201,0.697505,0.839240,0.743651,0.896659,0.998862,1.113278,0.890741,1.287897,1.390303,8.003096
min,15.000000,0.000000,0.000000,1.000000,1.000000,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000
25%,16.000000,2.000000,2.000000,1.000000,1.000000,0.000000,4.000000,3.000000,2.000000,1.000000,1.000000,3.000000,0.000000
50%,17.000000,3.000000,2.000000,1.000000,2.000000,0.000000,4.000000,3.000000,3.000000,1.000000,2.000000,4.000000,4.000000
75%,18.000000,4.000000,3.000000,2.000000,2.000000,0.000000,5.000000,4.000000,4.000000,2.000000,3.000000,5.000000,8.000000
max,22.000000,4.000000,4.000000,4.000000,4.000000,3.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,75.000000


In [29]:
X.describe(exclude=["number"])

,school,sex,address,famsize,Pstatus,Mjob,Fjob,reason,guardian,schoolsup,famsup,paid,activities,nursery,higher,internet,romantic
count,395,395,395,395,395,395,395,395,395,395,395,395,395,395,395,395,395
unique,2,2,2,2,2,5,5,4,3,2,2,2,2,2,2,2,2
top,GP,F,U,GT3,T,other,other,course,mother,no,yes,no,yes,yes,yes,yes,no
freq,349,208,307,281,354,141,217,145,273,344,242,214,201,314,375,329,263


Split train-test. Ratio 90:10.

In [31]:
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=TRAIN_SIZE, random_state=42, stratify=y)

X_train.shape, y_train.shape, X_test.shape, y_test.shape

((355, 30), (355,), (40, 30), (40,))

Save processed data.

In [32]:
X_train.to_csv(PROCESSED_DATA_DIR / 'X_train.csv', index=False)
X_test.to_csv(PROCESSED_DATA_DIR / 'X_test.csv', index=False)
y_train.to_csv(PROCESSED_DATA_DIR / 'y_train.csv', index=False)
y_test.to_csv(PROCESSED_DATA_DIR / 'y_test.csv', index=False)